In [ ]:
import json
import random
import re
import time
from datetime import date, datetime, timezone
from enum import Enum
from pathlib import Path
from typing import Any

import pandas as pd
from pydantic import BaseModel, Field
from pydantic_ai import Agent

from renewables_permitting.utils import (
    normalize_text,
    save_parquet,
    validate_required_columns,
)

BASE_URL = "https://www.boe.es/datosabiertos/api/boe/sumario"

# PROJECT_ROOT = Path(__file__).resolve().parents[2]  # fuera del notebook
PROJECT_ROOT = Path.cwd().parent  # dentro del notebook

DATA_DIR = PROJECT_ROOT / "data"

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

# BRONZE
BOE_DOCS_XML_DIR = BRONZE_DIR / "boe_docs_xml"


# SILVER
BOE_CANDIDATES_PATH = SILVER_DIR / "boe_candidates" / "boe_candidates_normalized.parquet"

BOE_CANDIDATES_DOCS_TEXT_PATH = SILVER_DIR / "boe_candidates_docs_text" / "boe_candidates_docs_text.parquet"

DIM_MUNICIPALITIES_PATH = SILVER_DIR / "dimensions" / "dim_municipalities.parquet"

SILVER_BOE_AI_DIR = SILVER_DIR / "boe_ai"

BOE_AI_EXTRACTIONS_PATH = SILVER_BOE_AI_DIR / "boe_ai_extractions.parquet"
LIFECYCLE_EVENTS_PATH = SILVER_BOE_AI_DIR / "lifecycle_events.parquet"
ADMINISTRATIVE_ACTIONS_PATH = SILVER_BOE_AI_DIR / "administrative_actions.parquet"
ASSET_MENTIONS_PATH = SILVER_BOE_AI_DIR / "asset_mentions.parquet"
ASSET_TECHNOLOGIES_PATH = SILVER_BOE_AI_DIR / "asset_technologies.parquet"
ASSET_PARTICIPANTS_PATH = SILVER_BOE_AI_DIR / "asset_participants.parquet"
ASSET_LOCATIONS_PATH = SILVER_BOE_AI_DIR / "asset_locations.parquet"
ASSET_ALIASES_PATH = SILVER_BOE_AI_DIR / "asset_aliases.parquet"
ASSET_RELATION_MENTIONS_PATH = SILVER_BOE_AI_DIR / "asset_relation_mentions.parquet"


# GOLD
PROJECT_GROUPS_PATH = GOLD_DIR / "project_groups.parquet"
PROJECT_ASSETS_PATH = GOLD_DIR / "project_assets.parquet"
PROJECT_TIMELINE_PATH = GOLD_DIR / "project_timeline.parquet"
PROJECT_STATUS_PATH = GOLD_DIR / "project_status.parquet"

In [2]:
def build_dim_municipalities(
    codine_path: Path,
    dictionary_path: Path,
) -> pd.DataFrame:
    """
    Construye la dimensión normalizada de municipios españoles a partir de
    ficheros de referencia del INE.

    La dimensión resultante se usa como tabla de referencia para resolver
    municipios mencionados en documentos del BOE. No es una tabla gold,
    porque no contiene información analítica de proyectos, sino datos
    administrativos de referencia.

    Parameters
    ----------
    codine_path : Path
        Ruta del fichero con correspondencia entre comunidad autónoma,
        provincia y sus códigos administrativos.
    dictionary_path : Path
        Ruta del fichero con el diccionario de municipios del INE.

    Returns
    -------
    pd.DataFrame
        DataFrame con códigos administrativos, nombres oficiales y campos
        normalizados para búsqueda determinista.
    """

    dim = pd.read_csv(codine_path, dtype=str)
    dicc = pd.read_csv(dictionary_path, dtype=str)

    required_dim_cols = {
        "codauto",
        "cpro",
        "comunidad_autonoma",
        "provincia",
    }

    required_dicc_cols = {
        "codauto",
        "cpro",
        "cmun",
        "nombre",
    }

    missing_dim_cols = required_dim_cols - set(dim.columns)
    missing_dicc_cols = required_dicc_cols - set(dicc.columns)

    if missing_dim_cols:
        raise ValueError(
            f"Faltan columnas obligatorias en {codine_path}: "
            f"{sorted(missing_dim_cols)}"
        )

    if missing_dicc_cols:
        raise ValueError(
            f"Faltan columnas obligatorias en {dictionary_path}: "
            f"{sorted(missing_dicc_cols)}"
        )

    dim = dim.copy()
    dicc = dicc.copy()

    for col in ["codauto", "cpro"]:
        dim[col] = dim[col].str.zfill(2)
        dicc[col] = dicc[col].str.zfill(2)

    dicc["cmun"] = dicc["cmun"].str.zfill(3)

    municipalities = dicc.merge(
        dim[
            [
                "codauto",
                "cpro",
                "comunidad_autonoma",
                "provincia",
            ]
        ],
        on=["codauto", "cpro"],
        how="left",
        validate="many_to_one",
    )

    municipalities = municipalities.rename(
        columns={
            "codauto": "cauto",
            "nombre": "municipio",
        }
    )

    municipalities["municipio_norm"] = municipalities["municipio"].map(normalize_text)
    municipalities["provincia_norm"] = municipalities["provincia"].map(normalize_text)
    municipalities["comunidad_autonoma_norm"] = municipalities[
        "comunidad_autonoma"
    ].map(normalize_text)

    municipalities["ine_municipality_code"] = (
        municipalities["cpro"] + municipalities["cmun"]
    )

    municipalities["ine_full_municipality_code"] = (
        municipalities["cauto"] + municipalities["cpro"] + municipalities["cmun"]
    )

    municipalities = municipalities[
        [
            "cauto",
            "comunidad_autonoma",
            "comunidad_autonoma_norm",
            "cpro",
            "provincia",
            "provincia_norm",
            "cmun",
            "ine_municipality_code",
            "ine_full_municipality_code",
            "municipio",
            "municipio_norm",
        ]
    ]

    municipalities = municipalities.sort_values(
        [
            "comunidad_autonoma",
            "provincia",
            "municipio",
        ]
    ).reset_index(drop=True)

    return municipalities

In [3]:
def validate_dim_municipalities(
    municipalities: pd.DataFrame,
) -> None:
    """
    Valida la dimensión de municipios construida a partir de referencias INE.

    La clave administrativa completa esperada es la combinación:
    cauto + cpro + cmun.

    Los nombres de municipio no deben ser únicos, porque existen municipios
    homónimos en distintas provincias o comunidades autónomas.

    Parameters
    ----------
    municipalities : pd.DataFrame
        Dimensión de municipios.

    Raises
    ------
    ValueError
        Si faltan columnas obligatorias, existen claves administrativas
        duplicadas o hay valores nulos en campos estructurales.
    """

    required_cols = {
        "cauto",
        "cpro",
        "cmun",
        "ine_municipality_code",
        "ine_full_municipality_code",
        "municipio",
        "municipio_norm",
        "provincia",
        "provincia_norm",
        "comunidad_autonoma",
        "comunidad_autonoma_norm",
    }

    missing_cols = required_cols - set(municipalities.columns)

    if missing_cols:
        raise ValueError(
            "Faltan columnas obligatorias en dim_municipalities: "
            f"{sorted(missing_cols)}"
        )

    key_cols = ["cauto", "cpro", "cmun"]

    null_key_rows = municipalities.loc[
        municipalities[key_cols].isna().any(axis=1)
    ]

    if not null_key_rows.empty:
        raise ValueError(
            "Existen municipios con cauto, cpro o cmun nulos."
        )

    duplicated_keys = municipalities.loc[
        municipalities.duplicated(subset=key_cols, keep=False)
    ]

    if not duplicated_keys.empty:
        raise ValueError(
            "Existen claves administrativas duplicadas en dim_municipalities "
            f"para la combinación {key_cols}."
        )

    duplicated_full_codes = municipalities.loc[
        municipalities["ine_full_municipality_code"].duplicated(keep=False)
    ]

    if not duplicated_full_codes.empty:
        raise ValueError(
            "Existen ine_full_municipality_code duplicados."
        )

    null_name_rows = municipalities.loc[
        municipalities[
            [
                "municipio",
                "municipio_norm",
                "provincia",
                "provincia_norm",
                "comunidad_autonoma",
                "comunidad_autonoma_norm",
            ]
        ].isna().any(axis=1)
    ]

    if not null_name_rows.empty:
        raise ValueError(
            "Existen municipios con nombres administrativos o normalizados nulos."
        )

## Prueba

In [4]:
municipios_ine_df = build_dim_municipalities(
    codine_path=BRONZE_DIR / "localizaciones_ine" / "codine_ccaaprovincia_20260614.csv",
    dictionary_path=BRONZE_DIR / "localizaciones_ine" / "diccionario26.csv",
)

validate_dim_municipalities(municipios_ine_df)

save_parquet(
    municipios_ine_df,
    DIM_MUNICIPALITIES_PATH,
)

municipios_ine_df.head(3)

,cauto,comunidad_autonoma,comunidad_autonoma_norm,cpro,provincia,provincia_norm,cmun,ine_municipality_code,ine_full_municipality_code,municipio,municipio_norm
0,01,Andalucía,andalucia,04,Almería,almeria,001,04001,0104001,Abla,abla
1,01,Andalucía,andalucia,04,Almería,almeria,002,04002,0104002,Abrucena,abrucena
2,01,Andalucía,andalucia,04,Almería,almeria,003,04003,0104003,Adra,adra


In [5]:
municipios_homonimos = (
    municipios_ine_df.loc[
        municipios_ine_df["municipio_norm"].duplicated(keep=False)
    ]
    .sort_values(["municipio_norm", "provincia", "municipio"])
)

municipios_homonimos.head(20)

,cauto,comunidad_autonoma,comunidad_autonoma_norm,cpro,provincia,provincia_norm,cmun,ine_municipality_code,ine_full_municipality_code,municipio,municipio_norm
6695,11,Extremadura,extremadura,10,Cáceres,caceres,023,10023,1110023,Arroyomolinos,arroyomolinos
7222,13,"Madrid, Comunidad de",madrid comunidad de,28,Madrid,madrid,015,28015,1328015,Arroyomolinos,arroyomolinos
6137,10,Comunitat Valenciana,comunitat valenciana,12,Castellón/Castelló,castellon castello,033,12033,1012033,Cabanes,cabanes
5357,09,Cataluña,cataluna,17,Girona,girona,030,17030,0917030,Cabanes,cabanes
416,01,Andalucía,andalucia,21,Huelva,huelva,018,21018,0121018,"Campillo, El",campillo el
3408,07,Castilla y León,castilla y leon,47,Valladolid,valladolid,031,47031,0747031,"Campillo, El",campillo el
4342,08,Castilla-La Mancha,castilla la mancha,16,Cuenca,cuenca,067,16067,0816067,Castejón,castejon
7508,15,"Navarra, Comunidad Foral de",navarra comunidad foral de,31,Navarra,navarra,070,31070,1531070,Castejón,castejon
1770,06,Cantabria,cantabria,39,Cantabria,cantabria,021,39021,0639021,Cieza,cieza
7407,14,"Murcia, Región de",murcia region de,30,Murcia,murcia,019,30019,1430019,Cieza,cieza
